In [3]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter
from matplotlib.gridspec import GridSpec
%matplotlib qt

In [4]:
def compute_pos(step_size, image_size):
    # Generate positions for updating symmetry image
    positions = []
    for j in range(image_size):
        for i in range(0, image_size, step_size):
            positions.append(np.array([i, j]))
    return np.array(positions)

# def convert_ij_to_pq(i, j, kernel_size):
#     # Convert (i, j) to (p, q) with kernel size
#     p = i + (kernel_size - 1) / 2
#     q = j + (kernel_size - 1) / 2
#     return [int(p), int(q)]

def update_symmetry(animated_image, image, i, j, step_size, image_size):
    # Update part of the animated image based on the original image
    updated_i = np.min([i + step_size, image_size])
    animated_image[j][i:updated_i] = image[j][i:updated_i]
    return animated_image

def update_kernal(display_image, p, q, kernel_size, kernal_thickness):
    # Draw the sliding kernel onto the display image
    display_image[q:q + kernal_thickness, p:p + kernel_size] = 255
    display_image[q + kernel_size - kernal_thickness:q + kernel_size, p:p + kernel_size] = 255
    display_image[q:q + kernel_size, p:p + kernal_thickness] = 255
    display_image[q:q + kernel_size, p + kernel_size - kernal_thickness:p + kernel_size] = 255
    return display_image

In [5]:
# Parameter settings
image_size = 128
step_size = 20
kernel_size = 27
kernal_thickness = 2

# example images, you can use your own images to substitute
image = np.random.randint(0, 256, (image_size, image_size), dtype=np.uint8)
image_with_border = np.random.randint(0, 256, (image_size + (kernel_size - 1), image_size + (kernel_size - 1)), dtype=np.uint8)

# Dynamically updated image
animated_image = np.full_like(image, 255)  # Initialize as all white (255)

# Image with the sliding kernel
display_image = image_with_border.copy()

# Create the figure and subplots
fig = plt.figure(figsize=(10, 6), dpi=100)
gs = GridSpec(1, 2, width_ratios=[4, 3])  # Left plot is wider than the right plot
ax1 = fig.add_subplot(gs[0])
ax2 = fig.add_subplot(gs[1])

# Adjust layout
ax1.axis("off")
ax2.axis("off")

# Left image (sliding kernel visualization)
im1 = ax1.imshow(display_image, cmap="gray", vmin=0, vmax=255)
# Right image (gradually revealed pixels)
im2 = ax2.imshow(animated_image, cmap="gray", vmin=0, vmax=255)

# Compute positions for the sliding kernel
positions = compute_pos(step_size, image_size)

def update(frame):
    global frame_idx, display_image, animated_image
    i, j = positions[frame]
    display_image = image_with_border.copy()
    animated_image = update_symmetry(animated_image, image, i, j, step_size, image_size)
    display_image = update_kernal(display_image, i, j, kernel_size, kernal_thickness)
    im2.set_array(animated_image)
    im1.set_array(display_image)
    return [im1, im2]

# Create the animation
ani = FuncAnimation(
    fig,
    update,
    frames=len(positions) - 1,
    interval=5,  # Interval between frames in milliseconds
    blit=True,
)

# Save the animation as a GIF
ani.save(
    "symmetry_animation.gif",
    writer=PillowWriter(fps=200),
    savefig_kwargs={"transparent": True, "pad_inches": 0},
)

plt.close(fig)